In [1]:
print("Hello")

Hello


In [ ]:
!pip install tensorflow scikit-learn pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.0 MB/s eta 0:00:00


In [6]:


import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout


df = pd.read_parquet(
    '/content/part-00000-2f3b7654-f2e6-47a7-8076-809f7c7a1346-c000.snappy.parquet'
)
# For CSV:
# df = pd.read_csv('/content/centralized_dataset.csv')

print("Original Dataset Shape:", df.shape)

required_columns = [
    'value',
    'Moving_avg',
    'Moving_std',
    'lag_1',
    'lag_2',
    'trend',
    'state_label'
]

df = df[required_columns]

print("Selected Dataset Shape:", df.shape)

# feature column definations
feature_cols = [
    'value',
    'Moving_avg',
    'Moving_std',
    'lag_1',
    'lag_2',
    'trend'
]

# numeric features
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=feature_cols)

# label encoding
label_encoder = LabelEncoder()
df['state_label'] = label_encoder.fit_transform(df['state_label'])

print("After Cleaning Shape:", df.shape)

# splitting the features and labels
X = df[feature_cols].astype('float32')
y = df['state_label'].astype('float32')

print("Final Features Shape:", X.shape)
print("Final Labels Shape:", y.shape)

# Testing and training split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshaping for GRU
X_train_gru = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_gru = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print("GRU Train Shape:", X_train_gru.shape)
print("GRU Test Shape :", X_test_gru.shape)

# class count determination
num_classes = len(np.unique(y))

# building the model
if num_classes == 2:
    output_units = 1
    output_activation = 'sigmoid'
    loss_function = 'binary_crossentropy'
else:
    output_units = num_classes
    output_activation = 'softmax'
    loss_function = 'sparse_categorical_crossentropy'

gru_model = Sequential([
    GRU(
        units=64,
        return_sequences=False,
        input_shape=(X_train_gru.shape[1], 1)
    ),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(output_units, activation=output_activation)
])

gru_model.compile(
    optimizer='adam',
    loss=loss_function,
    metrics=['accuracy']
)

gru_model.summary()

# Training the model
start_time = time.time()

history = gru_model.fit(
    X_train_gru,
    y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

training_time = time.time() - start_time

# prediction
y_pred_prob = gru_model.predict(X_test_gru)

if num_classes == 2:
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
else:
    y_pred = np.argmax(y_pred_prob, axis=1)

# Label Type
y_test_eval = y_test.astype(int)

# Evaluation
accuracy = accuracy_score(y_test_eval, y_pred)
precision = precision_score(
    y_test_eval,
    y_pred,
    average='weighted',
    zero_division=0
)
recall = recall_score(
    y_test_eval,
    y_pred,
    average='weighted',
    zero_division=0
)
f1 = f1_score(
    y_test_eval,
    y_pred,
    average='weighted',
    zero_division=0
)

# showing the results
print("\n===== CENTRALIZED GRU Performance =====")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("Training Time (seconds):", round(training_time, 2))

# Results saving
results = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1_Score": f1,
    "Training_Time_Seconds": training_time
}])

# results.to_csv("/content/centralized_gru_results.csv", index=False)

print("\nCentralized GRU results saved successfully.")

Original Dataset Shape: (1000000, 17)
Selected Dataset Shape: (1000000, 7)
After Cleaning Shape: (1000000, 7)
Final Features Shape: (1000000, 6)
Final Labels Shape: (1000000,)
Train Shape: (800000, 6)
Test Shape : (200000, 6)
GRU Train Shape: (800000, 6, 1)
GRU Test Shape : (200000, 6, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,043 (58.76 KB)

 Trainable params: 15,043 (58.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - accuracy: 0.7310 - loss: 0.5853 - val_accuracy: 0.8748 - val_loss: 0.3593
Epoch 2/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8490 - loss: 0.3538 - val_accuracy: 0.8804 - val_loss: 0.3069
Epoch 3/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 38s 6ms/step - accuracy: 0.8659 - loss: 0.3151 - val_accuracy: 0.8728 - val_loss: 0.2829
Epoch 4/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.8717 - loss: 0.2995 - val_accuracy: 0.9071 - val_loss: 0.2497
Epoch 5/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.8789 - loss: 0.2832 - val_accuracy: 0.8311 - val_loss: 0.3676
Epoch 6/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.8809 - loss: 0.2766 - val_accuracy: 0.8823 - val_loss: 0.2578
Epoch 7/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.8835 - loss: 0.2704 - val_accuracy: 0.8660 - val_loss: 0.2850
Epoch 8/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.8853 - loss: 0